# UBL 2.5 Sheets: Complete Revision Metadata & Export Links

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/investigate-oasis-sheets-aYOlG/notebooks/revision-metadata-and-export-links.ipynb)

## Purpose

Collect **all** revision metadata for both UBL 2.5 Google Sheets:

1. **v3 `revisions.list`** — gets ~25 "major" revisions with timestamps & authors
2. **v2 `revisions/{id}`** — for EVERY revision ID (1→max), gets `modifiedDate` + `exportLinks`

The export links from v2 are the **only proven method** (Method B) for downloading
revision-specific ODS content. v3 `files.export` ignores the revision parameter
for Google Sheets.

## Output

```
Drive: ubl-gc-revisions/
├── revision-index-ubl25_library.json     # All revisions with timestamps + exportLinks
├── revision-index-ubl25_documents.json
└── revision-index-summary.json           # Combined summary
```

With this data we can:
- Match revisions to CI workflow timestamps for correct ODS pairings
- Download any revision's ODS via its exportLink (proven Method B)
- See the full edit timeline with author attribution

## API Notes

| API | Endpoint | Returns | Revision-specific? |
|-----|----------|---------|--------------------|
| v3 | `revisions.list` | ~25 major revisions with timestamps | Metadata only |
| v2 | `revisions/{id}` | Single revision: modifiedDate + exportLinks | **YES — proven** |
| v3 | `files.export` | File export | **NO — broken for Sheets** |
| v2 | exportLinks URLs | ODS/CSV/XLSX download | **YES — proven** |

In [ ]:
# === Step 0: Auth ===
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

In [ ]:
# === Step 1: Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')

In [ ]:
# === Step 2: Configuration & Helpers ===
import json, time, hashlib
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from datetime import datetime

SHEETS = {
    'ubl25_library':   {'id': '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
                        'max_rev': 2005},
    'ubl25_documents': {'id': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
                        'max_rev': 2204},
}

# Known CI workflow run timestamps (UTC)
CI_RUNS = {
    'V1':  '2025-11-17T10:42:00Z',
    'V2':  '2025-11-19T09:15:00Z',
    'V3':  '2025-11-20T13:50:00Z',
    'V4':  '2025-11-20T14:05:00Z',
    'V5':  '2025-12-03T09:00:00Z',
    'V6':  '2026-01-21T16:38:00Z',
    'V7':  '2026-01-21T17:01:00Z',
    'V8':  '2026-01-21T19:26:00Z',
    'V9':  '2026-02-09T14:42:00Z',
    'V10': '2026-02-09T14:46:00Z',
}


def api_get_json(url, retries=4):
    """Authenticated JSON GET with retry + exponential backoff."""
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(retries):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=30) as resp:
                return resp.status, json.loads(resp.read())
        except HTTPError as e:
            body = e.read().decode(errors='replace')[:200]
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  retry({e.code}, wait={wait}s)...', end='')
                time.sleep(wait)
                continue
            return e.code, body
        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(2 ** (attempt + 1))
                continue
            return 0, str(exc)
    return 0, 'max retries exceeded'


print('Helpers ready')
print(f'Sheets: {list(SHEETS.keys())}')
print(f'CI runs: {list(CI_RUNS.keys())}')

## Step 3: v3 `revisions.list` — Major Revisions

Gets the ~25 "major" revisions that Google considers significant.
Each has: `id`, `modifiedTime`, `lastModifyingUser`, `size`.

In [ ]:
def list_revisions_v3(file_id):
    """Paginate through all revisions via Drive API v3."""
    all_revs = []
    page_token = None
    while True:
        url = (
            f'https://www.googleapis.com/drive/v3/files/{file_id}/revisions'
            f'?pageSize=1000'
            f'&fields=nextPageToken,revisions(id,modifiedTime,'
            f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,'
            f'size,exportLinks)'
        )
        if page_token:
            url += f'&pageToken={page_token}'

        status, data = api_get_json(url)
        if status != 200:
            print(f'  ERROR {status}: {str(data)[:200]}')
            break

        revs = data.get('revisions', [])
        all_revs.extend(revs)
        print(f'  Fetched {len(revs)} (total: {len(all_revs)})')

        page_token = data.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.5)

    return all_revs


v3_data = {}
for sheet_key, info in SHEETS.items():
    print(f'\n=== v3 revisions.list: {sheet_key} ===')
    revs = list_revisions_v3(info['id'])
    v3_data[sheet_key] = revs

    print(f'\n  Total: {len(revs)} major revisions')
    if revs:
        print(f'  First: id={revs[0]["id"]}, time={revs[0].get("modifiedTime", "?")}')
        print(f'  Last:  id={revs[-1]["id"]}, time={revs[-1].get("modifiedTime", "?")}')
        # Show revision ID range
        ids = [int(r['id']) for r in revs if r['id'].isdigit()]
        if ids:
            print(f'  ID range: {min(ids)} - {max(ids)}')
    time.sleep(1)

In [ ]:
# Show all v3 major revisions with timestamps
for sheet_key, revs in v3_data.items():
    print(f'\n{"="*70}')
    print(f'{sheet_key}: {len(revs)} major revisions from v3')
    print(f'{"="*70}')
    print(f'{"ID":>6}  {"Modified Time":30}  {"Author":30}  {"Size":>10}')
    print(f'{"-"*80}')
    for r in revs:
        rid = r.get('id', '?')
        mod = r.get('modifiedTime', '?')
        user = r.get('lastModifyingUser', {})
        author = user.get('displayName', user.get('emailAddress', '?'))
        size = r.get('size', '?')
        # Check if any CI run falls near this timestamp
        ci_match = ''
        for ver, ts in CI_RUNS.items():
            if mod and ts and mod[:16] <= ts[:16]:
                ci_match = f' <- before {ver}'
                break
        print(f'{rid:>6}  {mod:30}  {author:30}  {str(size):>10}{ci_match}')

    # Check if v3 returns exportLinks
    has_export_links = any(r.get('exportLinks') for r in revs)
    print(f'\n  v3 provides exportLinks: {has_export_links}')
    if has_export_links:
        sample = next(r for r in revs if r.get('exportLinks'))
        print(f'  Available formats:')
        for mime in sample['exportLinks']:
            print(f'    {mime}')

## Step 4: v2 Probe — Check Revision ID Accessibility

Before iterating all ~2005 IDs, test a few to confirm v2 can access them.
We test: rev 1 (first), a mid-range rev, the max rev, and a known Method B rev.

In [ ]:
def get_revision_v2(file_id, rev_id):
    """Get single revision metadata from Drive API v2.
    Returns (status, data_dict_or_error_string)."""
    url = f'https://www.googleapis.com/drive/v2/files/{file_id}/revisions/{rev_id}'
    return api_get_json(url)


# Probe tests
print('v2 Revision Accessibility Probe')
print('='*70)

test_ids = {
    'ubl25_library': ['1', '100', '500', '1000', '1843', '2005'],
    'ubl25_documents': ['1', '100', '500', '1000', '1793', '2204'],
}

for sheet_key, info in SHEETS.items():
    file_id = info['id']
    print(f'\n--- {sheet_key} ---')
    for rid in test_ids[sheet_key]:
        status, data = get_revision_v2(file_id, rid)
        if status == 200 and isinstance(data, dict):
            mod_date = data.get('modifiedDate', '?')
            has_links = bool(data.get('exportLinks'))
            n_formats = len(data.get('exportLinks', {}))
            print(f'  rev-{rid:>5}: OK  modified={mod_date}  '
                  f'exportLinks={n_formats} formats')
        else:
            print(f'  rev-{rid:>5}: HTTP {status} — {str(data)[:80]}')
        time.sleep(0.3)

## Step 5: Full v2 Scan — All Revision Metadata + Export Links

Iterate through ALL revision IDs (1 → max_rev) via v2.
For each accessible revision, collect:
- `modifiedDate` (timestamp)
- `lastModifyingUser` (author)
- `exportLinks` (download URLs)
- `fileSize`

This is ~2005 + ~2204 = ~4200 API calls at 0.3s each = ~21 minutes.

**Crash-resilient:** saves progress every 100 revisions. Re-run to resume.

In [ ]:
ODS_MIME = 'application/x-vnd.oasis.opendocument.spreadsheet'
ODS_MIME_ALT = 'application/vnd.oasis.opendocument.spreadsheet'


def extract_v2_revision_info(data):
    """Extract useful fields from a v2 revision response."""
    export_links = data.get('exportLinks', {})
    ods_link = export_links.get(ODS_MIME) or export_links.get(ODS_MIME_ALT)

    user = data.get('lastModifyingUser', {})

    return {
        'id': data.get('id'),
        'modifiedDate': data.get('modifiedDate'),
        'author': user.get('displayName', user.get('emailAddress')),
        'authorEmail': user.get('emailAddress'),
        'fileSize': data.get('fileSize'),
        'odsExportLink': ods_link,
        'exportFormats': list(export_links.keys()),
    }


def scan_all_revisions_v2(sheet_key, file_id, max_rev, save_path):
    """Iterate all revision IDs via v2, collecting metadata + exportLinks.
    Saves progress periodically. Returns the full index."""

    # Resume from saved progress
    if save_path.exists():
        index = json.loads(save_path.read_text())
        done_ids = {r['id'] for r in index.get('revisions', [])}
        print(f'  Resuming: {len(done_ids)} already done')
    else:
        index = {
            'sheet_key': sheet_key,
            'file_id': file_id,
            'max_rev': max_rev,
            'scan_started': datetime.utcnow().isoformat() + 'Z',
            'revisions': [],
            'missing_ids': [],
            'error_ids': [],
        }
        done_ids = set()

    found = len([r for r in index['revisions'] if r.get('modifiedDate')])
    missing = len(index.get('missing_ids', []))
    errors = len(index.get('error_ids', []))

    for rev_num in range(1, max_rev + 1):
        rid = str(rev_num)
        if rid in done_ids:
            continue

        status, data = get_revision_v2(file_id, rid)

        if status == 200 and isinstance(data, dict):
            info = extract_v2_revision_info(data)
            index['revisions'].append(info)
            done_ids.add(rid)
            found += 1

            if found % 50 == 0 or rev_num <= 5:
                print(f'  [{rev_num}/{max_rev}] found={found} '
                      f'miss={missing} err={errors} '
                      f'last={info["modifiedDate"]}')
        elif status in (400, 404):
            # Revision doesn't exist at this ID
            index['missing_ids'].append(rev_num)
            done_ids.add(rid)
            missing += 1
        else:
            # Unexpected error
            index['error_ids'].append({'id': rev_num, 'status': status,
                                       'error': str(data)[:100]})
            done_ids.add(rid)
            errors += 1
            print(f'  [{rev_num}] ERROR {status}: {str(data)[:80]}')

        # Save progress every 100 revisions
        if (found + missing + errors) % 100 == 0:
            index['scan_progress'] = f'{rev_num}/{max_rev}'
            index['stats'] = {'found': found, 'missing': missing, 'errors': errors}
            save_path.write_text(json.dumps(index, indent=1))

        time.sleep(0.3)

    # Final save
    index['scan_completed'] = datetime.utcnow().isoformat() + 'Z'
    index['stats'] = {'found': found, 'missing': missing, 'errors': errors}
    # Sort revisions by modifiedDate
    index['revisions'].sort(key=lambda r: r.get('modifiedDate', ''))
    save_path.write_text(json.dumps(index, indent=1))

    print(f'\n  DONE: {found} found, {missing} missing, {errors} errors')
    return index


print('Scanner ready')

In [ ]:
# Run the full scan for both sheets
v2_indices = {}

for sheet_key, info in SHEETS.items():
    save_path = DRIVE_DIR / f'revision-index-{sheet_key}.json'
    print(f'\n{"="*60}')
    print(f'Scanning {sheet_key}: revisions 1..{info["max_rev"]}')
    print(f'Save path: {save_path}')
    print(f'{"="*60}')

    index = scan_all_revisions_v2(
        sheet_key, info['id'], info['max_rev'], save_path
    )
    v2_indices[sheet_key] = index

print('\n\nAll sheets scanned!')

## Step 6: Analyze Results — Timeline & CI Matching

With timestamps for every accessible revision, we can:
1. Build a complete edit timeline
2. Find the exact revision that was current when each CI workflow ran
3. Identify the correct library+documents pairing for each CI version

In [ ]:
from datetime import datetime


def parse_ts(ts_str):
    """Parse ISO timestamp string to datetime."""
    if not ts_str:
        return None
    # Handle various formats
    for fmt in ['%Y-%m-%dT%H:%M:%S.%fZ', '%Y-%m-%dT%H:%M:%SZ',
                '%Y-%m-%dT%H:%M:%S.%f', '%Y-%m-%dT%H:%M:%S']:
        try:
            return datetime.strptime(ts_str, fmt)
        except ValueError:
            continue
    return None


def find_revision_at_time(revisions, target_time):
    """Find the latest revision that was saved BEFORE the target time.
    This is what a workflow would have downloaded."""
    target_dt = parse_ts(target_time)
    if not target_dt:
        return None

    best = None
    for r in revisions:
        mod_dt = parse_ts(r.get('modifiedDate'))
        if mod_dt and mod_dt <= target_dt:
            if best is None or mod_dt > parse_ts(best.get('modifiedDate')):
                best = r
    return best


# For each sheet, show timeline and CI matches
for sheet_key, index in v2_indices.items():
    revs = index.get('revisions', [])
    print(f'\n{"="*80}')
    print(f'{sheet_key}: {len(revs)} accessible revisions')
    print(f'{"="*80}')

    if not revs:
        print('  No revisions found!')
        continue

    # Show first and last
    print(f'  Earliest: id={revs[0]["id"]} at {revs[0].get("modifiedDate", "?")}')
    print(f'  Latest:   id={revs[-1]["id"]} at {revs[-1].get("modifiedDate", "?")}')

    # Show edit frequency by month
    months = {}
    for r in revs:
        mod = r.get('modifiedDate', '')
        if mod:
            month = mod[:7]  # YYYY-MM
            months[month] = months.get(month, 0) + 1
    print(f'\n  Edits by month:')
    for month in sorted(months):
        bar = '#' * min(months[month], 50)
        print(f'    {month}: {months[month]:4d} {bar}')

    # Find revision at each CI run time
    print(f'\n  Revision at each CI workflow run time:')
    print(f'  {"CI":>4}  {"Workflow Time":>24}  {"Rev ID":>7}  {"Rev Time":>28}  {"Author"}')
    print(f'  {"-"*100}')

    for ver, ci_time in CI_RUNS.items():
        match = find_revision_at_time(revs, ci_time)
        if match:
            print(f'  {ver:>4}  {ci_time:>24}  {match["id"]:>7}  '
                  f'{match.get("modifiedDate", "?"):>28}  '
                  f'{match.get("author", "?")}')
        else:
            print(f'  {ver:>4}  {ci_time:>24}  (no revision found before this time)')

In [ ]:
# === Build the definitive revision-to-CI mapping ===
# For each CI version, find the CORRECT library + documents revision pair

print('\n' + '='*80)
print('DEFINITIVE REVISION MAPPING')
print('='*80)
print()
print('For each CI workflow run, the revision that was current at that time:')
print()

mapping = {}
for ver, ci_time in CI_RUNS.items():
    entry = {'ci_time': ci_time}
    for sheet_key, index in v2_indices.items():
        revs = index.get('revisions', [])
        match = find_revision_at_time(revs, ci_time)
        role = 'library' if 'library' in sheet_key else 'documents'
        if match:
            entry[role] = {
                'rev_id': match['id'],
                'modified': match.get('modifiedDate'),
                'author': match.get('author'),
                'ods_export_link': match.get('odsExportLink'),
            }
        else:
            entry[role] = None
    mapping[ver] = entry

# Display
print(f'{"Ver":>4}  {"CI Time":>24}  {"Library Rev":>10}  {"Lib Modified":>28}  '
      f'{"Docs Rev":>10}  {"Docs Modified":>28}')
print('-' * 120)
for ver, m in mapping.items():
    lib = m.get('library', {})
    doc = m.get('documents', {})
    print(f'{ver:>4}  {m["ci_time"]:>24}  '
          f'{lib.get("rev_id", "?"):>10}  {lib.get("modified", "?"):>28}  '
          f'{doc.get("rev_id", "?"):>10}  {doc.get("modified", "?"):>28}')

# Compare with original mapping
ORIGINAL = {
    'V1':  {'lib': '1843', 'doc': '1793'},
    'V2':  {'lib': '1843', 'doc': '1793'},
    'V3':  {'lib': '1868', 'doc': '1803'},
    'V4':  {'lib': '1868', 'doc': '1983'},
    'V5':  {'lib': '1999', 'doc': '2190'},
    'V6':  {'lib': '1999', 'doc': '2190'},
    'V7':  {'lib': '2005', 'doc': '2190'},
    'V8':  {'lib': '2005', 'doc': '2200'},
    'V9':  {'lib': '2005', 'doc': '2204'},
    'V10': {'lib': '2005', 'doc': '2204'},
}

print(f'\n\nComparison with original mapping (based on ~25 major revisions):')
print(f'{"Ver":>4}  {"Old Lib":>8}  {"New Lib":>8}  {"Lib Changed?":>12}  '
      f'{"Old Doc":>8}  {"New Doc":>8}  {"Doc Changed?":>12}')
print('-' * 80)
for ver in ['V1','V2','V3','V4','V5','V6','V7','V8','V9','V10']:
    old = ORIGINAL[ver]
    new = mapping[ver]
    new_lib = new.get('library', {}).get('rev_id', '?')
    new_doc = new.get('documents', {}).get('rev_id', '?')
    lib_changed = 'CHANGED' if old['lib'] != new_lib else 'same'
    doc_changed = 'CHANGED' if old['doc'] != new_doc else 'same'
    print(f'{ver:>4}  {old["lib"]:>8}  {new_lib:>8}  {lib_changed:>12}  '
          f'{old["doc"]:>8}  {new_doc:>8}  {doc_changed:>12}')

In [ ]:
# === Show revisions around key transition points ===
# Focus on the periods where CI versions changed

WINDOWS = [
    ('V1->V2', '2025-11-17T00:00:00Z', '2025-11-19T12:00:00Z'),
    ('V2->V3', '2025-11-19T08:00:00Z', '2025-11-20T15:00:00Z'),
    ('V3->V4', '2025-11-20T13:00:00Z', '2025-11-20T15:00:00Z'),
    ('V4->V5', '2025-11-20T14:00:00Z', '2025-12-03T12:00:00Z'),
    ('V5->V6', '2025-12-03T08:00:00Z', '2026-01-21T17:00:00Z'),
    ('V6->V7', '2026-01-21T16:00:00Z', '2026-01-21T17:30:00Z'),
    ('V7->V8', '2026-01-21T17:00:00Z', '2026-01-21T20:00:00Z'),
    ('V8->V9', '2026-01-21T19:00:00Z', '2026-02-09T15:00:00Z'),
]

for label, start, end in WINDOWS:
    start_dt = parse_ts(start)
    end_dt = parse_ts(end)

    print(f'\n--- {label}: {start} to {end} ---')
    for sheet_key, index in v2_indices.items():
        role = 'LIB' if 'library' in sheet_key else 'DOC'
        revs_in_window = []
        for r in index.get('revisions', []):
            mod = parse_ts(r.get('modifiedDate'))
            if mod and start_dt <= mod <= end_dt:
                revs_in_window.append(r)

        if revs_in_window:
            for r in revs_in_window:
                print(f'  {role} rev-{r["id"]:>5}  {r["modifiedDate"]}  {r.get("author", "?")}')
        else:
            # Show the latest revision before the window
            before = find_revision_at_time(index.get('revisions', []), start)
            if before:
                print(f'  {role} (no edits in window, last was rev-{before["id"]} '
                      f'at {before["modifiedDate"]})')
            else:
                print(f'  {role} (no revisions found)')

## Step 7: Save Combined Summary

In [ ]:
# Save comprehensive summary
summary = {
    'generated': datetime.utcnow().isoformat() + 'Z',
    'description': 'Complete revision metadata for UBL 2.5 Google Sheets',
    'method': 'Drive API v3 revisions.list + v2 revisions/{id}',
    'sheets': {},
    'ci_revision_mapping': mapping,
    'ci_run_timestamps': CI_RUNS,
}

for sheet_key, index in v2_indices.items():
    revs = index.get('revisions', [])
    summary['sheets'][sheet_key] = {
        'file_id': index['file_id'],
        'total_revisions_found': len(revs),
        'total_missing': len(index.get('missing_ids', [])),
        'total_errors': len(index.get('error_ids', [])),
        'v3_major_revisions': len(v3_data.get(sheet_key, [])),
        'first_revision': revs[0] if revs else None,
        'last_revision': revs[-1] if revs else None,
    }

summary_path = DRIVE_DIR / 'revision-index-summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(f'Summary saved: {summary_path}')
print(f'  Size: {summary_path.stat().st_size:,} bytes')

# Also show sizes of the full index files
for sheet_key in SHEETS:
    idx_path = DRIVE_DIR / f'revision-index-{sheet_key}.json'
    if idx_path.exists():
        print(f'  {idx_path.name}: {idx_path.stat().st_size:,} bytes')

## Step 8: Export Links for Key Revisions

Show the actual ODS download URLs for the revisions identified above.
These can be used with `download-revision-ods.py` or directly with curl:

```bash
curl -H "Authorization: Bearer $TOKEN" -o rev-1843.ods "<export_link_url>"
```

In [ ]:
print('ODS Export Links for Key Revisions')
print('='*80)
print()

for ver, m in mapping.items():
    print(f'--- {ver} (CI: {m["ci_time"]}) ---')
    for role in ['library', 'documents']:
        info = m.get(role, {})
        if info:
            link = info.get('ods_export_link', 'N/A')
            print(f'  {role:>10} rev-{info["rev_id"]}: {info["modified"]}')
            if link and link != 'N/A':
                # Truncate for display
                print(f'             URL: {link[:100]}...')
            else:
                print(f'             URL: not available')
    print()

# Count unique export links across all revisions
for sheet_key, index in v2_indices.items():
    revs = index.get('revisions', [])
    has_link = sum(1 for r in revs if r.get('odsExportLink'))
    print(f'{sheet_key}: {has_link}/{len(revs)} revisions have ODS export links')

## What Next

With the revision index in hand:

1. **Download ODS** for any revision using its `odsExportLink` (Method B, proven)
2. **Pair correctly** — use timestamps to find the exact library+documents revision
   that was current when each CI workflow ran
3. **Convert to GC** — use the ODS→GC pipeline (Saxon + Crane) 
4. **Verify** — compare GC output against CI artifacts to confirm correct mapping

The key files saved to Drive:
- `revision-index-ubl25_library.json` — all library revision metadata + export links
- `revision-index-ubl25_documents.json` — all documents revision metadata + export links  
- `revision-index-summary.json` — combined summary with CI mapping